# 02b — Fetch 2-Year Transaction History

Fetches investment transactions from all connected brokerages for the past 2 years
using `investments/transactions/get`. Paginates through all results (max 500/page).

Saves:
- `transactions.json` — normalized flat list of all transactions
- `raw_plaid_responses/*.json` — raw Plaid API responses for each page

**Prerequisites:**
- Run notebook `01_connect_accounts.ipynb` first to generate `access_tokens.json`
- `PLAID_ENV=development` and valid `PLAID_CLIENT_ID` / `PLAID_SECRET` in `.env`

## Setup

In [ ]:
import json
import os
import sys
from datetime import datetime, date, timedelta
from collections import defaultdict

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

from fetch_transactions import fetch_all_transactions, TOKENS_FILE, TRANSACTIONS_FILE

if not os.path.exists(TOKENS_FILE):
    raise FileNotFoundError(
        "access_tokens.json not found. Run notebook 01 first to connect accounts."
    )

with open(TOKENS_FILE) as f:
    tokens = json.load(f)

print(f"✅ Found {len(tokens)} access token(s): {list(tokens.keys())}")
print(f"   Lookback: 2 years (730 days)")

## Fetch Transactions

In [ ]:
print("\n📊 Fetching 2 years of investment transactions...\n")
result = fetch_all_transactions()

transactions = result["transactions"]
raw_responses = result["raw_responses"]

print(f"\n{'═'*50}")
print(f"  Total transactions fetched: {len(transactions)}")
print(f"  Raw response pages stored:  {len(raw_responses)}")
print(f"{'═'*50}")

## Summary by Brokerage

In [ ]:
if transactions:
    df = pd.DataFrame(transactions)
    df["date"] = pd.to_datetime(df["date"])

    print("\n📊 Transactions by Brokerage:")
    brokerage_counts = df.groupby("brokerage").size().sort_values(ascending=False)
    for brokerage, count in brokerage_counts.items():
        print(f"   {brokerage:15s}: {count:4d} transactions")
else:
    print("No transactions found.")
    df = pd.DataFrame()

## Summary by Transaction Type

In [ ]:
if not df.empty:
    print("\n📊 Transactions by Type:")
    type_counts = df.groupby("type").size().sort_values(ascending=False)
    for txn_type, count in type_counts.items():
        print(f"   {txn_type:15s}: {count:4d}")

    print(f"\n📅 Date Range:")
    print(f"   Earliest: {df['date'].min().date()}")
    print(f"   Latest:   {df['date'].max().date()}")

    # Recent 30 days
    cutoff = pd.Timestamp.now() - pd.Timedelta(days=30)
    recent = df[df["date"] >= cutoff]
    print(f"\n📅 Last 30 Days: {len(recent)} transactions")

## Recent Transactions (Last 30 Days)

In [ ]:
if not df.empty:
    cutoff = pd.Timestamp.now() - pd.Timedelta(days=30)
    recent = df[df["date"] >= cutoff].sort_values("date", ascending=False)
    
    if not recent.empty:
        display_cols = ["date", "brokerage", "ticker", "type", "quantity", "price", "amount", "fees"]
        display_cols = [c for c in display_cols if c in recent.columns]
        print(f"Recent transactions (last 30 days):")
        recent[display_cols].head(20)
    else:
        print("No transactions in the last 30 days.")

## Transactions Saved

`transactions.json` has been saved by `fetch_all_transactions()`.
Raw Plaid responses are in `raw_plaid_responses/`.

These will be:
1. Synced to Supabase by `sync_to_supabase.py` (step 9 + 10)
2. Included in the Claude analysis payload in notebook 04